<a href="https://colab.research.google.com/github/zensworkspace001-afk/chinese-name-tagger/blob/main/colab_train_and_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 中文姓名切分模型 — Colab 訓練 + 測試

跑之前記得：**執行階段 → 變更執行階段類型 → 硬體加速器選 GPU (T4)**，不然下面訓練會用 CPU 跑很慢。

流程：
1. 檢查 GPU
2. 把專案檔案弄進 Colab（Google Drive 掛載，推薦）
3. 安裝套件
4. 偵測目前有哪些已訓練好的模型資料夾
5. 訓練
6. **訓練完成後把模型搬到 device（GPU），在該 device 上測試**
7. 也可以直接測試某個既有的模型資料夾（不重新訓練）
8. 把訓練好的模型存回 Google Drive（Colab 執行階段結束後本機檔案會消失）

## 1. 檢查 GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("沒偵測到 GPU —— 去『執行階段 > 變更執行階段類型』開 T4 GPU，不然訓練會很慢")

CUDA available: True
GPU: Tesla T4


## 2. 把專案檔案弄進 Colab

整個資料夾在你電腦上大約 4.6GB，主要是 11 個 `model_bert_v*` 模型資料夾（每個 ~388MB）。
**不需要全部上傳。** 建議只上傳：
- 程式碼：`surnames.py`, `names_pool.py`, `augment_data.py`, `fetch_real_corpus.py`, `fetch_wikinews.py`, `train_bert.py`, `predict_bert.py`, `requirements.txt`
- 資料：所有 `*.jsonl`（每個都在 MB 等級，很小）——包含新的 `diverse_train/val/test.jsonl`（data + real + augmented + wikinews 四個來源合併，訓練用這份）
- 如果你想直接測試既有模型而不是重新訓練：只上傳 `model_bert_v4`（README/CHANGELOG 裡目前唯一驗證有效的舊版本，~388MB，可以拿來跟這次新訓練的模型比較），不要整包 11 個都傳

最簡單的做法：把上面這些檔案放進 Google Drive 的某個資料夾（例如 `MyDrive/Deeplearning`），下面掛載 Drive 後直接讀。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

# 改成你在 Google Drive 裡放專案檔案的路徑
PROJECT_DIR = "/content/drive/MyDrive/Deeplearning"

os.chdir(PROJECT_DIR)
!ls

不想用 Google Drive 的話，改用直接上傳 zip（把上面列的檔案在本機打包成 `project.zip` 再上傳）：

```python
from google.colab import files
uploaded = files.upload()  # 選擇 project.zip
!unzip -q project.zip -d /content/project
import os; os.chdir("/content/project")
```

In [ ]:


import os

# Unzip the uploaded file into a new directory
!unzip -q /content/colab_upload.zip -d /content/project

# Change the current working directory to the project directory
os.chdir("/content/project")
!ls

diverse_test.jsonl   diverse_val.jsonl	requirements.txt  train_bert.py
diverse_train.jsonl  predict_bert.py	surnames.py


## 3. 安裝套件

In [ ]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 4. 偵測目前有哪些已訓練好的模型資料夾

跟 `app.py` 裡 `list_models()` 用同一套邏輯：找出資料夾裡有 `config.json` 的 `model_bert*` 目錄。
剛掛載完 Drive 時，這裡只會列出你自己上傳過去的（例如你只傳了 `model_bert_v4` 的話就只會看到它）；訓練完成後，新的輸出資料夾也會出現在這個清單。

In [ ]:
import glob


def list_models(base_dir="."):
    dirs = sorted(glob.glob(os.path.join(base_dir, "model_bert*")))
    return [d for d in dirs if os.path.isfile(os.path.join(d, "config.json"))]


available_models = list_models()
print("偵測到的已訓練模型資料夾：")
for d in available_models:
    print(" -", d)
if not available_models:
    print("(目前沒偵測到任何已訓練好的模型資料夾，訓練完後這裡會出現新的)")

## 5. 訓練

`PREFIX` 是資料檔名前綴（對應 `{PREFIX}_train/val/test.jsonl`）：
- **`diverse`（這次新的預設值）** — `data`（模板生成）+ `real`（維基百科弱標註，187 人擴充後）+ `augmented`（模板實體替換，新增了「A與B與C」「標籤：姓名」「姓名（括號）」三種句型）+ `wikinews`（維基新聞，83 位現代人物的真實新聞語體）四個來源合併，train 5322 筆、val 1422 筆、test 1459 筆
- `combo` — 舊資料，目前唯一驗證有效的 `model_bert_v4` 用的就是這份
- `final` — 舊的合併資料（`data`+`real`+`augmented`，沒有 `wikinews`）
- `augmented` / `real` / `wikinews` / `data` — 個別來源，細節見 `CHANGELOG.md` 跟各支 `fetch_*.py`/`augment_data.py` 檔案開頭的註解

`train_bert.py` 原本的裝置判斷只有 `mps`/`cpu`，沒有處理 `cuda`，在 Colab 上會白白浪費 GPU、退回用 CPU 訓練——已經在專案檔案裡修好了（優先 `cuda` > `mps` > `cpu`）。

**關於 `PATIENCE`（開發集連續幾次沒進步才把學習率減半）**：`CHANGELOG.md` 記錄過 `model_bert_v6` 把這個從 1 調成 2（更寬容），驗證 F1 創新高，但五篇獨立真實文件的表現反而輸給 patience=1 的 v4——這是目前唯一一個「歷史上已知會讓真實泛化能力變差」的訓練參數。這次資料量變大很多，結果不一定會重演，但**訓練完不能只看下面印出來的 F1 數字**，一定要用第 6 步、第 9 步的真實句子/文章去測，比對有沒有真的贏過 `model_bert_v4`。

In [6]:
PREFIX = "diverse"        # 訓練資料前綴（合併了 data+real+augmented+wikinews 四個來源）
OUT_DIR = "model_bert_colab"  # 訓練完成後的輸出資料夾名稱
EPOCHS = 30                # 從 20 調高——best checkpoint 還是照 val F1 挑，調高上限不會變差，只是給更多機會收斂
INIT_FROM = "hfl/chinese-roberta-wwm-ext"  # 從預訓練底座重新開始（不是 continued fine-tuning）
INITIAL_LR = 3e-5
FREEZE_BOTTOM_LAYERS = 0   # 從零訓練，不凍結
PATIENCE = 1               # v4 用的是 1；這裡調高到 2——注意上面 markdown 提過的風險，訓練完務必用真實句子驗證

!python train_bert.py {PREFIX} {OUT_DIR} {EPOCHS} {INIT_FROM} {INITIAL_LR} {FREEZE_BOTTOM_LAYERS} {PATIENCE}

從 hfl/chinese-roberta-wwm-ext 初始化，起始學習率 3.00e-05，凍結底部 0 層
Loading weights: 100% 197/197 [00:00<00:00, 25412.99it/s]
[transformers] BertForTokenClassification LOAD REPORT from: hfl/chinese-roberta-wwm-ext
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias              

## 6. 訓練完成後：把模型搬到 device，並在該 device 上測試

`predict_bert.py` 的 `load_model()` / `predict()` 也已經修好：`load_model(path, device=...)` 會把模型搬過去，`predict()` 會自動偵測模型現在在哪個 device、把輸入 tensor 也搬過去再跑（原本沒處理，模型跟輸入不同 device 會直接噴錯）。

In [8]:
from predict_bert import load_model, predict, extract_names, predict_document, split_sentences_keep_punct

device = torch.device(
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("使用裝置:", device)

model, tokenizer = load_model(OUT_DIR, device=device)
print("模型目前所在 device:", next(model.parameters()).device)

test_sentences = [
    "王小明昨天去看電影。",
    "歐陽鋒和洪七公在華山比武。",
    "老師陳美玲稱讚了學生林志豪的表現。",
    "諸葛亮向劉備獻上了妙計。",
    "這次記者會由張淑芬主持，來賓有李國瑞。",
]

for s in test_sentences:
    tagged = predict(s, model, tokenizer)
    names = extract_names(tagged)
    print(f"句子：{s}")
    print(f"  抽出姓名：{names}\n")

使用裝置: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

模型目前所在 device: cuda:0
句子：王小明昨天去看電影。
  抽出姓名：[('王', '小明')]

句子：歐陽鋒和洪七公在華山比武。
  抽出姓名：[]

句子：老師陳美玲稱讚了學生林志豪的表現。
  抽出姓名：[('陳', '美玲'), ('林', '志豪')]

句子：諸葛亮向劉備獻上了妙計。
  抽出姓名：[('諸', '葛亮')]

句子：這次記者會由張淑芬主持，來賓有李國瑞。
  抽出姓名：[('張', '淑芬'), ('李', '國瑞')]



In [7]:
import importlib, predict_bert
importlib.reload(predict_bert)
from predict_bert import load_model, predict, extract_names, predict_document, split_sentences_keep_punct

## 7. （可選）直接測試某個既有模型資料夾，不重新訓練

從第 4 步偵測到的 `available_models` 清單裡選一個，例如你上傳了 `model_bert_v4`。

In [ ]:
MODEL_TO_TEST = "model_bert_colab"  # 改成你要測試的資料夾名稱

assert MODEL_TO_TEST in [os.path.basename(d) for d in list_models()], (
    f"{MODEL_TO_TEST} 不在偵測到的資料夾清單裡，先確認有上傳/掛載到，或重新執行第 4 步"
)

model2, tokenizer2 = load_model(MODEL_TO_TEST, device=device)
print(f"已載入 {MODEL_TO_TEST}，device: {next(model2.parameters()).device}")

for s in test_sentences:
    tagged = predict(s, model2, tokenizer2)
    print(s, "->", extract_names(tagged))

NameError: name 'list_models' is not defined

## 8. 把訓練好的模型存回 Google Drive

Colab 執行階段結束/逾時後，`/content` 底下的檔案會消失。訓練完一定要存回 Drive，不然模型就沒了。

In [ ]:
import shutil

SAVE_TO_DRIVE = f"/content/drive/MyDrive/Deeplearning/{OUT_DIR}"
shutil.copytree(OUT_DIR, SAVE_TO_DRIVE, dirs_exist_ok=True)
print("已備份到", SAVE_TO_DRIVE)

已備份到 /content/drive/MyDrive/Deeplearning/model_bert_colab


## 9. （可選）貼自己的文章測試，含篇章級後處理

用 `predict_document`（全局擴散 + 局部擴散後處理，細節見 `predict_bert.py` 開頭註解）測整篇文章，而不是單一句子。

In [ ]:
my_text = """
把你自己的文章貼在這裡。
"""

sentences = split_sentences_keep_punct(my_text)
results = predict_document(sentences, model, tokenizer)
for s, tagged, names in results:
    print(s, "->", names)

In [9]:
!zip -r model_bert_colab.zip model_bert_colab

  adding: model_bert_colab/ (stored 0%)
  adding: model_bert_colab/config.json (deflated 56%)
  adding: model_bert_colab/tokenizer.json (deflated 75%)
  adding: model_bert_colab/model.safetensors (deflated 7%)
  adding: model_bert_colab/tokenizer_config.json (deflated 43%)


In [10]:
from google.colab import files
files.download('model_bert_colab.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>